In [1]:
"""
Approach: Multi-Task Spatio-Temporal EEG-to-Text Generation
- Architecture: GCN-GRU Encoder (using Granger Causality graphs) + Luong Attention Decoder.
- Auxiliary: Multi-label classification heads for Visual Metadata (Colors/Objects) to boost context.
- Training: Uncertainty-Weighted Loss (Auto-balancing), Diversity Regularization, and Scheduled Teacher Forcing.
- Inference: Beam Search decoding for robust, grammatically correct sentence generation.
"""

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate
import os

# ==================================================================================
# CONFIGURATION
# ==================================================================================

H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

BATCH_SIZE = 16
EPOCHS = 30 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# Dimensions
NUM_COLORS = 9       
NUM_OBJECTS = 6      
COLOR_NAMES = ["Black", "Blue", "Brown", "Green", "Grey", "Orange", "Red", "White", "Yellow"]
OBJECT_NAMES = ["Animal", "Building", "Food", "Nature", "Person", "Vehicle"]

# ==================================================================================
# FIXED DATA SPLIT (NO LEAKAGE)
# ==================================================================================

def create_stratified_split(total_samples, group_size=5):
    """
    Split data ensuring groups of 5 consecutive samples stay together.
    For each group: 3 train, 1 val, 1 test
    """
    num_groups = total_samples // group_size
    train_indices = []
    val_indices = []
    test_indices = []
    
    for group_idx in range(num_groups):
        start_idx = group_idx * group_size
        group_indices = list(range(start_idx, start_idx + group_size))
        
        # Deterministic split within each group
        train_indices.extend(group_indices[:3])  # First 3 for training
        val_indices.append(group_indices[3])     # 4th for validation
        test_indices.append(group_indices[4])    # 5th for testing
    
    # Handle remainder if total_samples not divisible by 5
    remainder = total_samples % group_size
    if remainder > 0:
        start_idx = num_groups * group_size
        remainder_indices = list(range(start_idx, total_samples))
        
        if remainder >= 3:
            train_indices.extend(remainder_indices[:3])
            if remainder >= 4:
                val_indices.append(remainder_indices[3])
            if remainder == 5:
                test_indices.append(remainder_indices[4])
        else:
            train_indices.extend(remainder_indices)
    
    print(f"Split Statistics:")
    print(f"  Total Groups: {num_groups}")
    print(f"  Train: {len(train_indices)} samples ({len(train_indices)/total_samples*100:.1f}%)")
    print(f"  Val:   {len(val_indices)} samples ({len(val_indices)/total_samples*100:.1f}%)")
    print(f"  Test:  {len(test_indices)} samples ({len(test_indices)/total_samples*100:.1f}%)")
    
    return train_indices, val_indices, test_indices

# ==================================================================================
# GRANGER CAUSALITY
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            min_len = 20
            if len(ts_i) < min_len or len(ts_j) < min_len:
                causality_matrix[i, j] = 0.0
                continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0:
        edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)

# ==================================================================================
# DATASET
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden


class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        # encoder_outputs: [seq_len, batch, enc_dim]
        # decoder_hidden: [1, batch, dec_dim]
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)


class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        self.color_processor = nn.Sequential(
            nn.Linear(num_colors, 64),
            nn.ReLU(),
            nn.Linear(64, color_feature_dim)
        )
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, object_feature_dim)
        )
        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        color_input = metadata[:, :NUM_COLORS].float()
        object_input = metadata[:, NUM_COLORS:].float()
        color_vec = self.color_processor(color_input)
        object_vec = self.object_processor(object_input)
        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)

        rnn_input = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)


class DiversityLoss(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
    
    def forward(self, logits):
        probs = F.softmax(logits, dim=-1)
        avg_probs = probs.mean(dim=(0, 1))
        uniform = torch.ones_like(avg_probs) / self.vocab_size
        kl_div = F.kl_div(avg_probs.log(), uniform, reduction='batchmean')
        return kl_div


class UncertaintyWeightedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_var_text = nn.Parameter(torch.zeros(1))
        self.log_var_color = nn.Parameter(torch.zeros(1))
        self.log_var_object = nn.Parameter(torch.zeros(1))
    
    def forward(self, loss_t, loss_c, loss_o):
        precision_t = torch.exp(-self.log_var_text)
        precision_c = torch.exp(-self.log_var_color)
        precision_o = torch.exp(-self.log_var_object)
        
        loss = (precision_t * loss_t + self.log_var_text +
                precision_c * loss_c + self.log_var_color +
                precision_o * loss_o + self.log_var_object)
        return loss


class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_feature_dim=32, object_feature_dim=32, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects, color_feature_dim, object_feature_dim)
        
        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                               meta_features_dim, dec_layers, pad_id, dropout)

        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)

        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:]

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input, decoder_hidden, encoder_outputs, meta_features, global_eeg_context
            )
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2), pred_color, pred_object

# ==================================================================================
# BEAM SEARCH INFERENCE
# ==================================================================================
def beam_search_decoder(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=3, max_len=30):
    model.eval()
    
    with torch.no_grad():
        eeg_signal = eeg_signal.unsqueeze(0).to(device)
        meta_signal = meta_signal.unsqueeze(0).to(device)
        
        encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
        meta_features = model.meta_encoder(meta_signal)
        
        decoder_hidden = model.decoder.init_hidden(encoder_hidden)
        
        hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        
        meta_preds = model.meta_head(global_eeg_context)
        pred_color_ids = (torch.sigmoid(meta_preds[0, :model.num_colors]) > 0.5).nonzero(as_tuple=True)[0].tolist()
        pred_object_ids = (torch.sigmoid(meta_preds[0, model.num_colors:]) > 0.5).nonzero(as_tuple=True)[0].tolist()

        # Beam: (score, input_id, hidden, sequence)
        beams = [(0.0, SOS_ID, decoder_hidden, [])]
        
        for _ in range(max_len):
            candidates = []
            for score, input_id, hidden, seq in beams:
                if len(seq) > 0 and seq[-1] == EOS_ID:
                    candidates.append((score, input_id, hidden, seq))
                    continue
                
                token_tensor = torch.tensor([input_id], device=device)
                prediction, new_hidden, _ = model.decoder(
                    token_tensor, hidden, encoder_outputs, meta_features, global_eeg_context
                )
                
                log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for k in range(beam_width):
                    next_score = score + topk_probs[k].item()
                    next_id = topk_ids[k].item()
                    candidates.append((next_score, next_id, new_hidden, seq + [next_id]))
            
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(seq[-1] == EOS_ID for _, _, _, seq in beams if len(seq) > 0):
                break

        best_score, _, _, best_seq = beams[0]
        if best_seq and best_seq[-1] == EOS_ID:
            best_seq = best_seq[:-1]
            
        predicted_text = tokenizer.decode(best_seq, skip_special_tokens=True)
        return predicted_text, pred_color_ids, pred_object_ids

# ==================================================================================
# TRAINING AND EVALUATION
# ==================================================================================

def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion,
                   object_criterion, diversity_criterion, uncertainty_loss, granger_edge_index, 
                   granger_edge_attr, diversity_weight=0.01, teacher_forcing_ratio=0.5):
    model.train()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        optimizer.zero_grad()

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=teacher_forcing_ratio
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        # LOSS BOOSTING: Force model to pay 2x attention to text errors
        loss_t_boosted = loss_t * 2.0

        loss_main = uncertainty_loss(loss_t_boosted, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

        progress_bar.set_postfix(loss=loss.item(), txt=loss_t.item())

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n

@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr, diversity_weight=0.01):
    model.eval()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_t_boosted = loss_t * 2.0
        loss_main = uncertainty_loss(loss_t_boosted, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n

# ==================================================================================
# MAIN EXECUTION
# ==================================================================================

if __name__ == "__main__":
    # 1. Setup Dataset & Split
    print("--- Loading Dataset ---")
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    
    # Use the new Fixed Data Split
    train_idx, val_idx, test_idx = create_stratified_split(len(dataset))
    
    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)
    test_ds = Subset(dataset, test_idx)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    # 2. Setup Graph
    print("Creating Granger Causality matrix...")
    try:
        eeg_b, _, _ = next(iter(train_loader))
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        num_channels = eeg_b.shape[1]
        granger_edge_index, granger_edge_attr = add_self_loops(
            granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels, fill_value=1.0
        )
        granger_edge_index = granger_edge_index.to(torch.long).to(device)
        granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
    except Exception as e:
        print(f"Error creating Granger matrix: {e}. Using fallback.")
        num_channels = 62
        edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
        granger_edge_index = edge_index.to(torch.long).to(device)
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device)

    # 3. Setup Model
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        dropout=0.2,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2
    ).to(device)

    object_pos_weight = torch.tensor([3.1176, 5.6667, 6.1066, 1.1021, 2.0905, 7.5366]).to(device)
    color_pos_weight = torch.tensor([3.1543, 1.2764, 3.9645, 1.7888, 0.8301, 11.2807, 5.2780, 1.0408, 4.4054]).to(device)

    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    color_criterion = nn.BCEWithLogitsLoss(pos_weight=color_pos_weight)
    object_criterion = nn.BCEWithLogitsLoss(pos_weight=object_pos_weight)
    diversity_criterion = DiversityLoss(TEXT_VOCAB_SIZE).to(device)
    uncertainty_loss = UncertaintyWeightedLoss().to(device)

    optimizer = AdamW(list(model.parameters()) + list(uncertainty_loss.parameters()), lr=3e-5, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)
    
    DIVERSITY_WEIGHT = 0.01
    best_val_loss = float('inf')

    # 4. Training Loop
    print("\n--- Starting Training ---")
    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()

        # Scheduled Teacher Forcing: Decay from 1.0 -> 0.0
        tf_ratio = max(0.0, 1.0 - (epoch / 20)) 

        train_components, train_loss = train_one_epoch(
            model, train_loader, optimizer,
            text_criterion, color_criterion, object_criterion,
            diversity_criterion, uncertainty_loss,
            granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT,
            teacher_forcing_ratio=tf_ratio
        )
        
        val_components, val_loss = evaluate(
            model, val_loader,
            text_criterion, color_criterion, object_criterion,
            diversity_criterion, uncertainty_loss,
            granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
        )

        scheduler.step(val_loss)
        
        print(f'\nEpoch: {epoch:02} | TF Ratio: {tf_ratio:.2f} | Time: {int(time.time() - start_time)}s')
        print(f'\tTrain Loss: {train_loss:.4f} | Text: {train_components["text"]:.4f}')
        print(f'\t  Val Loss: {val_loss:.4f} | Text: {val_components["text"]:.4f}')
        
        with torch.no_grad():
            print(f'\tTask Weights: Text={torch.exp(-uncertainty_loss.log_var_text).item():.3f}, '
                  f'Color={torch.exp(-uncertainty_loss.log_var_color).item():.3f}, '
                  f'Obj={torch.exp(-uncertainty_loss.log_var_object).item():.3f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'eeg-text-phases6-improved.pt')
            print(f"\t-> Val loss decreased. Saved model.")

    # 5. Inference & Evaluation
    print("\n--- Starting Evaluation with Beam Search ---")
    model.load_state_dict(torch.load('eeg-text-phases6-improved.pt', map_location=device))
    
    predictions = []
    references = []
    
    num_samples_eval = min(len(test_ds), 50)
    for i in range(num_samples_eval):
        eeg_sample, meta_sample, true_text_ids = test_ds[i]
        
        # Ground Truth
        true_text = tokenizer.decode(true_text_ids.tolist(), skip_special_tokens=True)
        references.append(true_text)

        # Prediction with Beam Search
        pred_text, _, _ = beam_search_decoder(
            model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr, beam_width=3
        )
        predictions.append(pred_text)
        
        if i < 5:
            print(f"\nSample {i+1}:")
            print(f"GT:   {true_text}")
            print(f"Pred: {pred_text}")

    # Compute Metrics
    try:
        bleu = hf_evaluate.load('bleu')
        results = bleu.compute(predictions=predictions, references=[[r] for r in references])
        print(f"\nBLEU Score: {results['bleu']:.4f}")
    except Exception as e:
        print(f"Metric Error: {e}")

    print("\n--- Complete ---")

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Using device: cuda
--- Loading Dataset ---
Split Statistics:
  Total Groups: 5600
  Train: 16800 samples (60.0%)
  Val:   5600 samples (20.0%)
  Test:  5600 samples (20.0%)
Creating Granger Causality matrix...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(



--- Starting Training ---


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


...

Training:   0%|          | 0/1050 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/350 [00:00<?, ?it/s]


Epoch: 13 | TF Ratio: 0.35 | Time: 385s
	Train Loss: 6.1648 | Text: 2.6890
	  Val Loss: 7.9858 | Text: 4.0209
	Task Weights: Text=0.696, Color=1.032, Obj=0.950
	-> Val loss decreased. Saved model.


Training:   0%|          | 0/1050 [00:00<?, ?it/s]

KeyboardInterrupt: 